# Adult sex CMAB fairness

This notebook runs the Adult Income experiments with sex as the sensitive attribute in the contextual multi-armed bandit framework

The objective is to compare three families of contextual bandit algorithms:

- **LinUCB family**: LinUCB and FairLinUCB;
- **Linear Thompson Sampling family**: LinTS and FairLinTS;
- **EXP4 family**: EXP4 and FairEXP4.

For each family, the notebook evaluates three levels of fairness intervention:

- **Pre-processing**: uniform weighting versus group-label reweighting;
- **In-processing**: standard policy versus demographic-parity-aware policy;
- **Post-processing**: group-specific threshold calibration on held-out data.

The binary action is interpreted as the predicted Adult income class. The reward is equal to 1 when the selected action matches the observed label and 0 otherwise. Therefore, average reward is equivalent to accuracy in this offline classification-derived bandit setting.

The main utility metrics are average reward and cumulative prediction error. Fairness is assessed using Demographic Parity Gap, Equalized Odds Gap, and UtilityGap. Temporal figures report the evolution of these metrics over the learning horizon.

## Imports and configuration

This section loads the project modules, defines the run mode, output directories, random seeds, horizon, policy-family hyperparameters, preprocessing settings, and post-processing settings.

For reproducibility, the notebook can either regenerate the full benchmark or reload cached outputs.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import pandas as pd
from dataclasses import replace
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root from the current notebook location.

    This lets the notebook run both from VS Code and from Jupyter, whether the
    current working directory is the project root or a subfolder such as notebooks/.
    """
    current = Path.cwd().resolve() if start is None else Path(start).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src" / "fair_bandits").exists():
            return candidate
        if (candidate / "fair_bandits").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


from fair_bandits.config import build_config

from fair_bandits.data import (
    load_adult,
    prepare_adult_sex_splits,
)

from fair_bandits.experiments import (
    AdultBanditParams,
    load_adult_family_outputs,
    run_adult_family_benchmark,
    train_adult_expert_pool,
)

from fair_bandits.io import (
    export_adult_final_summary_tables,
    export_adult_significance_tables,
)

from fair_bandits.metrics import normalize_metric_columns

from fair_bandits.plots.adult_cmab import (
    plot_adult_family_figure_set,
    plot_adult_final_dot_plots,
    plot_adult_postprocessing_over_horizon,
    plot_adult_linucb_inprocessing_average_reward,
    plot_adult_linucb_postprocessing_average_reward
)

from fair_bandits.plots import plot_real_dataset_tradeoff_set

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
RUN_MODE = "full"  # use "full" for thesis results
RUN_BENCHMARK = True
FORCE_RERUN = False

try:
    CFG = build_config(
        run_mode="full" if RUN_MODE == "full" else "dev",
    )
except TypeError:
    CFG = build_config(
        "full" if RUN_MODE == "full" else "dev",
    )

RESULTS_ROOT = Path(CFG.results_dir)
OUTPUT_ROOT = RESULTS_ROOT / "adult_sex_cmab"
RUN_DIR = OUTPUT_ROOT / RUN_MODE
FIG_DIR = OUTPUT_ROOT / "final_figures"
TABLE_DIR = OUTPUT_ROOT / "overleaf_tables"

ADULT_CSV_CANDIDATES = [
    PROJECT_ROOT / "data_cache" / "adult.csv",
    PROJECT_ROOT / "data_cache" / "adult.data",
    PROJECT_ROOT / "notebooks" / "data_cache" / "adult.csv",
    PROJECT_ROOT / "notebooks" / "data_cache" / "adult.data",
    PROJECT_ROOT / "data" / "adult.csv",
    PROJECT_ROOT / "data" / "adult.data",
    PROJECT_ROOT / "datasets" / "adult.csv",
    PROJECT_ROOT / "datasets" / "adult.data",
    Path.home() / "Downloads" / "adult.csv",
    Path.home() / "Downloads" / "adult.data",
]

ADULT_CSV_PATH = next(
    (
        path
        for path in ADULT_CSV_CANDIDATES
        if path.exists()
    ),
    None,
)

if ADULT_CSV_PATH is None:
    print("No local Adult CSV/data file found.")
    print("Loading Adult from OpenML through fair_bandits.data.load_adult().")

    adult_df = load_adult()

    print("Downloaded Adult dataframe:", adult_df.shape)
    print("Columns:", adult_df.columns.tolist())

else:
    print("ADULT_CSV_PATH:", ADULT_CSV_PATH)

print("ADULT_CSV_PATH:", ADULT_CSV_PATH)

PREPROCESSINGS = [
    "uniform",
    "reweigh_group_label",
]

PREPROCESSING_LABELS = {
    "uniform": "Uniform",
    "reweigh_group_label": "Reweighting",
}

POLICY_FAMILIES = {
    "linucb": {
        "label": "LinUCB",
        "policies": ["LinUCB", "FairLinUCB"],
        "postprocessed_policy": "FairLinUCB+PP",
    },
    "linear_ts": {
        "label": "Linear Thompson Sampling",
        "policies": ["LinTS", "FairLinTS"],
        "postprocessed_policy": "FairLinTS+PP",
    },
    "exp4": {
        "label": "EXP4",
        "policies": ["EXP4", "FairEXP4"],
        "postprocessed_policy": "FairEXP4+PP",
    },
}

POLICY_LABELS = {
    "LinUCB": "LinUCB",
    "FairLinUCB": "FairLinUCB",
    "FairLinUCB+PP": "FairLinUCB+PP",
    "LinTS": "LinTS",
    "FairLinTS": "FairLinTS",
    "FairLinTS+PP": "FairLinTS+PP",
    "EXP4": "EXP4",
    "FairEXP4": "FairEXP4",
    "FairEXP4+PP": "FairEXP4+PP",
}

if RUN_MODE == "dev":
    N_SEEDS = 2
    T_MAX = 1500
    LOG_EVERY = 25
    N_EXPERTS = 8
    EXPERT_BOOTSTRAP_SIZE = 6000
else:
    N_SEEDS = 50
    T_MAX = 30000
    LOG_EVERY = 50
    N_EXPERTS = 20
    EXPERT_BOOTSTRAP_SIZE = 12000

SEEDS = list(range(N_SEEDS))

TEST_SIZE = 0.20
CALIBRATION_SIZE_WITHIN_REMAINING = 0.20

LAMBDA_RIDGE = 10.0
ALPHA_LINUCB = 1.5
TS_V = 0.25

EXP4_GAMMA = 0.07
EXP4_ETA = None

DP_TAU = 0.02
DP_LAMBDA = 2.0
BETA_SMOOTH = 1.0
MIN_GROUP_COUNT = 20

MAX_ACCURACY_DROP = 0.01
THRESHOLD_GRID_SIZE = 31

print("RUN_MODE:", RUN_MODE)
print("RUN_DIR:", RUN_DIR)
print("FIG_DIR:", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)
print("N_SEEDS:", N_SEEDS)
print("T_MAX:", T_MAX)
print("RUN_BENCHMARK:", RUN_BENCHMARK)

## Load and prepare Adult sex dataset

The Adult dataset is loaded either from a local CSV/data file or, if no local file is found, through the existing `load_adult()` helper.

The dataset is then prepared with sex as the sensitive attribute. The target is encoded as a binary income label, the feature matrix is one-hot encoded and standardized, and the data are split into train, calibration, and test subsets. The split is stratified by group-label strata so that both sensitive-group and label distributions remain stable across subsets.

In [ ]:
adult_data = prepare_adult_sex_splits(
    adult_csv_path=ADULT_CSV_PATH,
    in_memory_frames=globals(),
    test_size=TEST_SIZE,
    calibration_size_within_remaining=CALIBRATION_SIZE_WITHIN_REMAINING,
)

print("\nTrain target proportion:")
display(
    pd.Series(adult_data.y_train)
    .value_counts(normalize=True)
    .rename("proportion")
)

print("\nTrain group proportion:")
display(
    pd.Series(adult_data.g_train)
    .value_counts(normalize=True)
    .rename("proportion")
)

print("\nTrain group × target:")
display(
    pd.crosstab(
        adult_data.g_train,
        adult_data.y_train,
        normalize="index",
    ).round(4)
)

In [ ]:
from fair_bandits.experiments import (
    tune_random_forest_hyperparameters,
    run_random_forest_benchmark,
)

RF_PARAMS, adult_rf_cv = tune_random_forest_hyperparameters(
    X_train=adult_data.X_train,
    y_train=adult_data.y_train,
    seed=42,
    n_estimators=500,
    cv_folds=3,
)

display(adult_rf_cv.head(12))
print(RF_PARAMS)

adult_rf_df = run_random_forest_benchmark(
    X_train=adult_data.X_train,
    y_train=adult_data.y_train,
    g_train=adult_data.g_train,
    X_test=adult_data.X_test,
    y_test=adult_data.y_test,
    g_test=adult_data.g_test,
    seeds=SEEDS,
    preprocessings=PREPROCESSINGS,
    params=RF_PARAMS,
    output_path=RUN_DIR / "random_forest_endpoint.csv",
)

display(adult_rf_df.groupby("preprocessing")[[
    "average_reward", "DP_gap", "EO_gap", "TPR_gap", "FPR_gap", "UtilityGap"
]].agg(["mean", "std"]))

## Train EXP4 expert pool

The EXP4 family requires expert advice. When the benchmark is regenerated, a supervised expert pool is trained on the Adult training set and converted into action-advice matrices for the train, calibration, and test sets.

When cached benchmark outputs are loaded, this step is skipped because the stored EXP4 results already contain the trajectories and final metrics.

In [ ]:
expert_pool = train_adult_expert_pool(
    adult_data.X_train,
    adult_data.y_train,
    n_experts=N_EXPERTS,
    bootstrap_size=EXPERT_BOOTSTRAP_SIZE,
    seed=2026,
)

ADVICE_TRAIN = expert_pool.predict_advice(adult_data.X_train)
ADVICE_CAL = expert_pool.predict_advice(adult_data.X_cal)
ADVICE_TEST = expert_pool.predict_advice(adult_data.X_test)

print("ADVICE_TRAIN:", ADVICE_TRAIN.shape)
print("ADVICE_CAL:", ADVICE_CAL.shape)
print("ADVICE_TEST:", ADVICE_TEST.shape)

## Run or load Adult policy-family benchmarks

This section runs or loads the Adult-sex contextual bandit experiments for the three policy families.

For LinUCB and LinTS, reweighting affects the update weights. For EXP4, reweighting affects the sampled training stream. This preserves the behaviour of the original notebooks while using a common module-based workflow.

Each family is evaluated across both preprocessing settings, all random seeds, and the full learning horizon. The outputs are concatenated into shared temporal, endpoint, post-processing, and parameter dataframes.

In [ ]:
params = AdultBanditParams(
    d=adult_data.X_train.shape[1],
    t_max=T_MAX,
    log_every=LOG_EVERY,
    alpha_linucb=ALPHA_LINUCB,
    ts_v=TS_V,
    lambda_ridge=LAMBDA_RIDGE,
    exp4_gamma=EXP4_GAMMA,
    exp4_eta=EXP4_ETA,
    n_experts=N_EXPERTS,
    dp_tau=DP_TAU,
    dp_lambda=DP_LAMBDA,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    max_accuracy_drop=MAX_ACCURACY_DROP,
    threshold_grid_size=THRESHOLD_GRID_SIZE,
)

all_temporal = []
all_endpoint = []
all_postproc = []
all_parameters = []

for family, family_config in POLICY_FAMILIES.items():
    print("\nFamily:", family)
    family_run_dir = RUN_DIR / family

    if RUN_BENCHMARK:
        temporal_df, endpoint_df, postproc_df, parameters_df = run_adult_family_benchmark(
            family=family,
            run_dir=family_run_dir,
            X_train=adult_data.X_train,
            y_train=adult_data.y_train,
            g_train=adult_data.g_train,
            X_cal=adult_data.X_cal,
            y_cal=adult_data.y_cal,
            g_cal=adult_data.g_cal,
            X_test=adult_data.X_test,
            y_test=adult_data.y_test,
            g_test=adult_data.g_test,
            advice_train=ADVICE_TRAIN if family == "exp4" else None,
            advice_cal=ADVICE_CAL if family == "exp4" else None,
            advice_test=ADVICE_TEST if family == "exp4" else None,
            preprocessings=PREPROCESSINGS,
            policies=family_config["policies"],
            seeds=SEEDS,
            params=params,
            force_rerun=FORCE_RERUN,
        )
    else:
        temporal_df, endpoint_df, postproc_df, parameters_df = load_adult_family_outputs(
            family=family,
            run_dir=family_run_dir,
        )

    all_temporal.append(temporal_df)
    all_endpoint.append(endpoint_df)
    all_postproc.append(postproc_df)
    all_parameters.append(parameters_df)

adult_temporal_df = normalize_metric_columns(pd.concat(all_temporal, ignore_index=True))
adult_endpoint_df = normalize_metric_columns(pd.concat(all_endpoint, ignore_index=True))
adult_postproc_df = normalize_metric_columns(pd.concat(all_postproc, ignore_index=True))
adult_parameters_df = pd.concat(all_parameters, ignore_index=True)

print("\nTemporal:", adult_temporal_df.shape)
print("Endpoint:", adult_endpoint_df.shape)
print("Post-processing:", adult_postproc_df.shape)
print("Parameters:", adult_parameters_df.shape)

## Validate benchmark outputs

This section checks that the expected families, policies, preprocessing settings, seeds, and final horizons are present in the loaded or newly generated results.

These checks are useful before generating figures and statistical tables, because missing cached files or incomplete runs would otherwise produce misleading summaries.

In [ ]:
print("Families:", sorted(adult_temporal_df["family"].unique()))
print("Policies:", sorted(adult_temporal_df["policy"].unique()))
print("Preprocessings:", sorted(adult_temporal_df["preprocessing"].unique()))
print("Seeds:", adult_temporal_df["seed"].nunique())
print("Final t values:", sorted(adult_endpoint_df["t"].unique()))

expected_endpoint_rows = (
    sum(len(config["policies"]) for config in POLICY_FAMILIES.values())
    * len(PREPROCESSINGS)
    * len(SEEDS)
)

expected_postproc_rows = (
    len(POLICY_FAMILIES)
    * len(PREPROCESSINGS)
    * len(SEEDS)
    * 2
)

print("Expected endpoint rows:", expected_endpoint_rows)
print("Observed endpoint rows:", len(adult_endpoint_df))
print("Expected post-processing rows:", expected_postproc_rows)
print("Observed post-processing rows:", len(adult_postproc_df))

display(adult_endpoint_df.head())
display(adult_postproc_df.head())

In [ ]:
# ============================================================
# Parameters needed for the ablation only
# Does NOT run the main benchmark
# ============================================================

params = AdultBanditParams(
    d=adult_data.X_train.shape[1],
    t_max=T_MAX,
    log_every=LOG_EVERY,
    alpha_linucb=ALPHA_LINUCB,
    ts_v=TS_V,
    lambda_ridge=LAMBDA_RIDGE,
    exp4_gamma=EXP4_GAMMA,
    exp4_eta=EXP4_ETA,
    n_experts=N_EXPERTS,
    dp_tau=DP_TAU,
    dp_lambda=DP_LAMBDA,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    max_accuracy_drop=MAX_ACCURACY_DROP,
    threshold_grid_size=THRESHOLD_GRID_SIZE,
)

print("RUN_DIR =", RUN_DIR)
print("Adult train =", adult_data.X_train.shape)
print("Seeds =", len(SEEDS))
print("Preprocessings =", PREPROCESSINGS)
print("T_MAX =", params.t_max)

In [ ]:
# ============================================================
# Adult — full ablation on LinUCB backbone
# Missing part only:
# standard LinUCB held-out evaluation + LinUCB+PP
# ============================================================

import json
import numpy as np
import pandas as pd

from fair_bandits.experiments.adult_runner import (
    run_adult_family_trajectory,
)
from fair_bandits.experiments.adult_scoring import (
    adult_linucb_score_table,
)
from fair_bandits.postprocessing import (
    actions_from_group_thresholds,
    optimize_group_thresholds,
)
from fair_bandits.metrics import (
    normalize_metric_columns,
    summarize_classification_bandit,
)

ABLATION_DIR = RUN_DIR / "ablation"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

STANDARD_ABLATION_PATH = (
    ABLATION_DIR
    / "adult_linucb_standard_holdout_postprocessing.csv"
)

STANDARD_ABLATION_PARAMETERS_PATH = (
    ABLATION_DIR
    / "adult_linucb_standard_postprocessing_parameters.csv"
)

ABLATION_FORCE_RERUN = False


# ------------------------------------------------------------
# Reload previous partial results if they exist
# ------------------------------------------------------------

if (
    STANDARD_ABLATION_PATH.exists()
    and not ABLATION_FORCE_RERUN
):
    adult_linucb_standard_ablation_df = pd.read_csv(
        STANDARD_ABLATION_PATH
    )
else:
    adult_linucb_standard_ablation_df = pd.DataFrame()


if (
    STANDARD_ABLATION_PARAMETERS_PATH.exists()
    and not ABLATION_FORCE_RERUN
):
    adult_linucb_standard_ablation_parameters_df = (
        pd.read_csv(
            STANDARD_ABLATION_PARAMETERS_PATH
        )
    )
else:
    adult_linucb_standard_ablation_parameters_df = (
        pd.DataFrame()
    )


# ------------------------------------------------------------
# Determine which seed/preprocessing combinations are complete
# ------------------------------------------------------------

done = set()

if not adult_linucb_standard_ablation_df.empty:

    for (
        seed,
        preprocessing,
    ), group in (
        adult_linucb_standard_ablation_df
        .groupby(
            ["seed", "preprocessing"]
        )
    ):

        policies_present = set(
            group["policy"].astype(str)
        )

        if {
            "LinUCB",
            "LinUCB+PP",
        }.issubset(
            policies_present
        ):
            done.add(
                (
                    int(seed),
                    str(preprocessing),
                )
            )


total = (
    len(SEEDS)
    * len(PREPROCESSINGS)
)

completed = len(done)

print(
    f"Already complete: "
    f"{completed}/{total}"
)


# ------------------------------------------------------------
# Run only missing standard-LinUCB held-out evaluations
# ------------------------------------------------------------

for seed in SEEDS:

    for preprocessing in PREPROCESSINGS:

        key = (
            int(seed),
            str(preprocessing),
        )

        if key in done:

            print(
                "Cached ablation:",
                key,
            )

            continue


        print(
            "Running ablation:",
            key,
        )


        # Train ONLY standard LinUCB
        _, linucb_policy = (
            run_adult_family_trajectory(
                family="linucb",

                X_train=adult_data.X_train,
                y_train=adult_data.y_train,
                g_train=adult_data.g_train,

                advice_train=None,

                seed=int(seed),

                policy_name="LinUCB",

                preprocessing=str(
                    preprocessing
                ),

                params=params,
            )
        )


        # ----------------------------------------------------
        # Frozen policy -> calibration set
        # ----------------------------------------------------

        calibration_table = (
            adult_linucb_score_table(
                linucb_policy,

                adult_data.X_cal,
                adult_data.y_cal,
                adult_data.g_cal,

                fair=False,
            )
        )


        # ----------------------------------------------------
        # Frozen policy -> test set
        # ----------------------------------------------------

        test_table = (
            adult_linucb_score_table(
                linucb_policy,

                adult_data.X_test,
                adult_data.y_test,
                adult_data.g_test,

                fair=False,
            )
        )


        # ----------------------------------------------------
        # Learn group thresholds ONLY on calibration set
        # ----------------------------------------------------

        (
            thresholds,
            calibration_raw,
            calibration_postprocessed,
        ) = optimize_group_thresholds(

            calibration_table,

            metric_fn=
                summarize_classification_bandit,

            max_accuracy_drop=
                params.max_accuracy_drop,

            threshold_grid_size=
                params.threshold_grid_size,
        )


        # Threshold = 0 reproduces original LinUCB decision
        zero_thresholds = {

            str(group): 0.0

            for group in sorted(
                np.unique(
                    np.asarray(
                        adult_data.g_test
                    ).astype(str)
                )
            )
        }


        raw_actions = (
            actions_from_group_thresholds(
                test_table,
                zero_thresholds,
            )
        )


        postprocessed_actions = (
            actions_from_group_thresholds(
                test_table,
                thresholds,
            )
        )


        # Remove possible incomplete previous result
        if not adult_linucb_standard_ablation_df.empty:

            keep = ~(
                (
                    adult_linucb_standard_ablation_df[
                        "seed"
                    ].astype(int)
                    == int(seed)
                )
                &
                (
                    adult_linucb_standard_ablation_df[
                        "preprocessing"
                    ].astype(str)
                    == str(preprocessing)
                )
            )

            adult_linucb_standard_ablation_df = (
                adult_linucb_standard_ablation_df
                .loc[keep]
                .copy()
            )


        new_rows = []


        for policy_name, actions in [

            (
                "LinUCB",
                raw_actions,
            ),

            (
                "LinUCB+PP",
                postprocessed_actions,
            ),

        ]:

            new_rows.append(
                {
                    "family":
                        "linucb",

                    "seed":
                        int(seed),

                    "preprocessing":
                        str(preprocessing),

                    "policy":
                        policy_name,

                    "t":
                        int(params.t_max),

                    **summarize_classification_bandit(
                        adult_data.y_test,
                        actions,
                        adult_data.g_test,
                    ),
                }
            )


        adult_linucb_standard_ablation_df = (
            pd.concat(
                [
                    adult_linucb_standard_ablation_df,
                    pd.DataFrame(
                        new_rows
                    ),
                ],
                ignore_index=True,
            )
        )


        # ----------------------------------------------------
        # Save thresholds for reproducibility
        # ----------------------------------------------------

        parameter_row = {

            "family":
                "linucb",

            "seed":
                int(seed),

            "preprocessing":
                str(preprocessing),

            "thresholds_json":
                json.dumps(
                    thresholds,
                    sort_keys=True,
                ),

            "calibration_raw_DP_gap":
                calibration_raw[
                    "DP_gap"
                ],

            "calibration_postproc_DP_gap":
                calibration_postprocessed[
                    "DP_gap"
                ],

            "calibration_raw_EO_gap":
                calibration_raw[
                    "EO_gap"
                ],

            "calibration_postproc_EO_gap":
                calibration_postprocessed[
                    "EO_gap"
                ],

            "calibration_raw_accuracy":
                calibration_raw[
                    "accuracy"
                ],

            "calibration_postproc_accuracy":
                calibration_postprocessed[
                    "accuracy"
                ],
        }


        if not (
            adult_linucb_standard_ablation_parameters_df.empty
        ):

            keep = ~(
                (
                    adult_linucb_standard_ablation_parameters_df[
                        "seed"
                    ].astype(int)
                    == int(seed)
                )
                &
                (
                    adult_linucb_standard_ablation_parameters_df[
                        "preprocessing"
                    ].astype(str)
                    == str(preprocessing)
                )
            )

            adult_linucb_standard_ablation_parameters_df = (
                adult_linucb_standard_ablation_parameters_df
                .loc[keep]
                .copy()
            )


        adult_linucb_standard_ablation_parameters_df = (
            pd.concat(
                [
                    adult_linucb_standard_ablation_parameters_df,
                    pd.DataFrame(
                        [parameter_row]
                    ),
                ],
                ignore_index=True,
            )
        )


        # ----------------------------------------------------
        # Save after every completed seed/preprocessing
        # ----------------------------------------------------

        adult_linucb_standard_ablation_df.to_csv(
            STANDARD_ABLATION_PATH,
            index=False,
        )

        adult_linucb_standard_ablation_parameters_df.to_csv(
            STANDARD_ABLATION_PARAMETERS_PATH,
            index=False,
        )


        completed += 1
        done.add(key)

        print(
            f"Completed ablation "
            f"{completed}/{total}"
        )


adult_linucb_standard_ablation_df = (
    normalize_metric_columns(
        adult_linucb_standard_ablation_df
    )
)


print(
    "\nStandard LinUCB held-out ablation:"
)

display(
    adult_linucb_standard_ablation_df
    .groupby(
        [
            "policy",
            "preprocessing",
        ]
    )[
        [
            "average_reward",
            "DP_gap",
            "EO_gap",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
)

In [ ]:
# ============================================================
# Reload existing held-out post-processing results
# No benchmark is rerun
# ============================================================

import pandas as pd

adult_postproc_df = pd.concat(
    [
        pd.read_csv(
            RUN_DIR / "linucb" / "postprocessing.csv"
        ),
        pd.read_csv(
            RUN_DIR / "linear_ts" / "postprocessing.csv"
        ),
        pd.read_csv(
            RUN_DIR / "exp4" / "postprocessing.csv"
        ),
    ],
    ignore_index=True,
)

adult_postproc_df = normalize_metric_columns(
    adult_postproc_df
)

print(
    "adult_postproc_df:",
    adult_postproc_df.shape
)

print(
    "\nPolicies available:"
)

print(
    adult_postproc_df["policy"]
    .value_counts()
)

In [ ]:
# ============================================================
# Adult — assemble complete 8-condition ablation
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import t


# Existing FairLinUCB held-out results
fair_linucb_holdout_df = (
    adult_postproc_df[
        (
            adult_postproc_df[
                "family"
            ].astype(str)
            == "linucb"
        )
        &
        (
            adult_postproc_df[
                "policy"
            ].astype(str).isin(
                [
                    "FairLinUCB",
                    "FairLinUCB+PP",
                ]
            )
        )
    ]
    .copy()
)


# New standard-LinUCB held-out results
standard_linucb_holdout_df = (
    adult_linucb_standard_ablation_df
    .copy()
)


adult_ablation_df = pd.concat(
    [
        standard_linucb_holdout_df,
        fair_linucb_holdout_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Map each condition to the factorial ablation
# ------------------------------------------------------------

CONFIG_MAP = {

    (
        "LinUCB",
        "uniform",
    ):
        "None",

    (
        "LinUCB",
        "reweigh_group_label",
    ):
        "Pre only",

    (
        "FairLinUCB",
        "uniform",
    ):
        "In only",

    (
        "LinUCB+PP",
        "uniform",
    ):
        "Post only",

    (
        "FairLinUCB",
        "reweigh_group_label",
    ):
        "Pre + In",

    (
        "LinUCB+PP",
        "reweigh_group_label",
    ):
        "Pre + Post",

    (
        "FairLinUCB+PP",
        "uniform",
    ):
        "In + Post",

    (
        "FairLinUCB+PP",
        "reweigh_group_label",
    ):
        "Pre + In + Post",
}


adult_ablation_df[
    "configuration"
] = [

    CONFIG_MAP[
        (
            str(policy),
            str(preprocessing),
        )
    ]

    for policy, preprocessing in zip(
        adult_ablation_df[
            "policy"
        ],
        adult_ablation_df[
            "preprocessing"
        ],
    )
]


CONFIG_ORDER = [
    "None",
    "Pre only",
    "In only",
    "Post only",
    "Pre + In",
    "Pre + Post",
    "In + Post",
    "Pre + In + Post",
]


adult_ablation_df[
    "configuration"
] = pd.Categorical(

    adult_ablation_df[
        "configuration"
    ],

    categories=
        CONFIG_ORDER,

    ordered=True,
)


# ------------------------------------------------------------
# Check: exactly 50 seeds for each of 8 conditions
# ------------------------------------------------------------

counts = (
    adult_ablation_df

    .groupby(
        "configuration",
        observed=False,
    )[
        "seed"
    ]

    .agg(
        [
            "size",
            "nunique",
        ]
    )
)


display(counts)


assert (
    counts["size"]
    == 50
).all()

assert (
    counts["nunique"]
    == 50
).all()


# ------------------------------------------------------------
# Summary with 95% Student-t CI
# ------------------------------------------------------------

ABLATION_METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
    "TPR_gap",
    "FPR_gap",
    "UtilityGap",
]


summary_rows = []


for configuration, group in (
    adult_ablation_df
    .groupby(
        "configuration",
        observed=False,
    )
):

    n = len(group)

    t_crit = t.ppf(
        0.975,
        df=n - 1,
    )


    row = {
        "configuration":
            str(configuration),

        "n":
            n,
    }


    for metric in ABLATION_METRICS:

        mean = group[
            metric
        ].mean()

        sd = group[
            metric
        ].std(
            ddof=1
        )

        ci95 = (
            t_crit
            * sd
            / np.sqrt(n)
        )


        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95

        row[
            f"{metric}_low"
        ] = (
            mean
            - ci95
        )

        row[
            f"{metric}_high"
        ] = (
            mean
            + ci95
        )


    summary_rows.append(
        row
    )


adult_ablation_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


adult_ablation_summary_df[
    "configuration"
] = pd.Categorical(

    adult_ablation_summary_df[
        "configuration"
    ],

    categories=
        CONFIG_ORDER,

    ordered=True,
)


adult_ablation_summary_df = (
    adult_ablation_summary_df

    .sort_values(
        "configuration"
    )

    .reset_index(
        drop=True
    )
)


display(
    adult_ablation_summary_df[
        [
            "configuration",

            "average_reward_mean",
            "average_reward_ci95",

            "DP_gap_mean",
            "DP_gap_ci95",

            "EO_gap_mean",
            "EO_gap_ci95",
        ]
    ]
    .round(4)
)


ABLATION_TABLE_PATH = (
    TABLE_DIR
    / "adult_linucb_full_ablation_summary.csv"
)


adult_ablation_summary_df.to_csv(
    ABLATION_TABLE_PATH,
    index=False,
)


print(
    "Saved:",
    ABLATION_TABLE_PATH,
)

In [ ]:
# Save complete seed-level ablation results
ABLATION_SEEDLEVEL_PATH = (
    TABLE_DIR
    / "adult_linucb_full_ablation_seedlevel.csv"
)

adult_ablation_df.to_csv(
    ABLATION_SEEDLEVEL_PATH,
    index=False,
)

print("Saved:", ABLATION_SEEDLEVEL_PATH)

In [ ]:
# ============================================================
# Adult — factorial ablation significance analysis
# 2 × 2 × 2 Pre / In / Post
#
# Planned comparisons = the 12 edges of the factorial cube
# Paired Wilcoxon tests across the same 50 seeds
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon


# ------------------------------------------------------------
# Reload seed-level results if necessary
# ------------------------------------------------------------

ABLATION_SEEDLEVEL_PATH = (
    TABLE_DIR
    / "adult_linucb_full_ablation_seedlevel.csv"
)

if "adult_ablation_df" not in globals():

    adult_ablation_df = pd.read_csv(
        ABLATION_SEEDLEVEL_PATH
    )


# ------------------------------------------------------------
# The 12 factorial edge contrasts
# ------------------------------------------------------------

CONTRASTS = [

    # ----------------------
    # Effect of PRE
    # ----------------------

    {
        "intervention": "Pre",
        "context": "None",
        "before": "None",
        "after": "Pre only",
    },

    {
        "intervention": "Pre",
        "context": "In",
        "before": "In only",
        "after": "Pre + In",
    },

    {
        "intervention": "Pre",
        "context": "Post",
        "before": "Post only",
        "after": "Pre + Post",
    },

    {
        "intervention": "Pre",
        "context": "In + Post",
        "before": "In + Post",
        "after": "Pre + In + Post",
    },


    # ----------------------
    # Effect of IN
    # ----------------------

    {
        "intervention": "In",
        "context": "None",
        "before": "None",
        "after": "In only",
    },

    {
        "intervention": "In",
        "context": "Pre",
        "before": "Pre only",
        "after": "Pre + In",
    },

    {
        "intervention": "In",
        "context": "Post",
        "before": "Post only",
        "after": "In + Post",
    },

    {
        "intervention": "In",
        "context": "Pre + Post",
        "before": "Pre + Post",
        "after": "Pre + In + Post",
    },


    # ----------------------
    # Effect of POST
    # ----------------------

    {
        "intervention": "Post",
        "context": "None",
        "before": "None",
        "after": "Post only",
    },

    {
        "intervention": "Post",
        "context": "Pre",
        "before": "Pre only",
        "after": "Pre + Post",
    },

    {
        "intervention": "Post",
        "context": "In",
        "before": "In only",
        "after": "In + Post",
    },

    {
        "intervention": "Post",
        "context": "Pre + In",
        "before": "Pre + In",
        "after": "Pre + In + Post",
    },
]


METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


# ------------------------------------------------------------
# Holm-Bonferroni
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    p_values = np.where(
        np.isfinite(p_values),
        p_values,
        1.0,
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty(
        len(p_values),
        dtype=float,
    )

    previous = 0.0

    for rank, index in enumerate(order):

        value = (
            len(p_values)
            - rank
        ) * p_values[index]

        value = min(
            max(
                value,
                previous,
            ),
            1.0,
        )

        adjusted[index] = value

        previous = value

    return adjusted


# ------------------------------------------------------------
# Run paired contrasts
# ------------------------------------------------------------

rows = []


for contrast in CONTRASTS:

    before_df = (
        adult_ablation_df[
            adult_ablation_df[
                "configuration"
            ].astype(str)
            == contrast["before"]
        ]
        .set_index("seed")
        .sort_index()
    )

    after_df = (
        adult_ablation_df[
            adult_ablation_df[
                "configuration"
            ].astype(str)
            == contrast["after"]
        ]
        .set_index("seed")
        .sort_index()
    )


    common_seeds = (
        before_df.index
        .intersection(
            after_df.index
        )
    )

    assert len(common_seeds) == 50


    for metric in METRICS:

        before_values = (
            before_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )

        after_values = (
            after_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )


        raw_difference = (
            after_values
            - before_values
        )


        # Benefit-oriented effect:
        #
        # reward: higher = better
        # DP/EO: lower = better
        if metric == "average_reward":

            benefit_difference = (
                after_values
                - before_values
            )

        else:

            benefit_difference = (
                before_values
                - after_values
            )


        if np.allclose(
            before_values,
            after_values,
        ):

            statistic = 0.0
            p_value = 1.0

        else:

            result = wilcoxon(
                before_values,
                after_values,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto",
            )

            statistic = (
                result.statistic
            )

            p_value = (
                result.pvalue
            )


        rows.append(
            {
                "intervention":
                    contrast[
                        "intervention"
                    ],

                "context":
                    contrast[
                        "context"
                    ],

                "before":
                    contrast[
                        "before"
                    ],

                "after":
                    contrast[
                        "after"
                    ],

                "metric":
                    metric,

                "before_mean":
                    before_values.mean(),

                "after_mean":
                    after_values.mean(),

                "raw_difference_after_minus_before":
                    raw_difference.mean(),

                # Positive = improvement,
                # whatever the metric
                "benefit_difference":
                    benefit_difference.mean(),

                "wilcoxon_statistic":
                    statistic,

                "p_raw":
                    p_value,
            }
        )


adult_ablation_significance_df = (
    pd.DataFrame(
        rows
    )
)


# ------------------------------------------------------------
# Holm correction across ALL 36 planned tests
# ------------------------------------------------------------

adult_ablation_significance_df[
    "p_holm_global"
] = holm_adjust(

    adult_ablation_significance_df[
        "p_raw"
    ]
)


adult_ablation_significance_df[
    "significant_global_0.05"
] = (

    adult_ablation_significance_df[
        "p_holm_global"
    ]

    < 0.05
)


# ------------------------------------------------------------
# Also calculate Holm correction separately within each metric
# Useful for supplementary interpretation
# ------------------------------------------------------------

adult_ablation_significance_df[
    "p_holm_within_metric"
] = np.nan


for metric in METRICS:

    mask = (
        adult_ablation_significance_df[
            "metric"
        ]
        == metric
    )

    adult_ablation_significance_df.loc[
        mask,
        "p_holm_within_metric",
    ] = holm_adjust(

        adult_ablation_significance_df.loc[
            mask,
            "p_raw",
        ]
    )


adult_ablation_significance_df[
    "significant_within_metric_0.05"
] = (

    adult_ablation_significance_df[
        "p_holm_within_metric"
    ]

    < 0.05
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    adult_ablation_significance_df[
        [
            "intervention",
            "context",
            "before",
            "after",
            "metric",
            "before_mean",
            "after_mean",
            "benefit_difference",
            "p_raw",
            "p_holm_global",
            "significant_global_0.05",
        ]
    ].round(5)
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

ABLATION_STATS_PATH = (
    TABLE_DIR
    / "adult_linucb_full_ablation_wilcoxon_holm.csv"
)

adult_ablation_significance_df.to_csv(
    ABLATION_STATS_PATH,
    index=False,
)

print(
    "Saved:",
    ABLATION_STATS_PATH,
)

In [ ]:
# ============================================================
# Adult — 3-panel ablation figure
#
# A: change in average reward vs None
# B: improvement in DP gap vs None
# C: improvement in EO gap vs None
#
# Positive values ALWAYS mean improvement
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t


CONFIG_ORDER = [
    "Pre only",
    "In only",
    "Post only",
    "Pre + In",
    "Pre + Post",
    "In + Post",
    "Pre + In + Post",
]


baseline = (
    adult_ablation_df[
        adult_ablation_df[
            "configuration"
        ].astype(str)
        == "None"
    ]
    .set_index("seed")
    .sort_index()
)


delta_rows = []


for configuration in CONFIG_ORDER:

    config_df = (
        adult_ablation_df[
            adult_ablation_df[
                "configuration"
            ].astype(str)
            == configuration
        ]
        .set_index("seed")
        .sort_index()
    )


    common_seeds = (
        baseline.index
        .intersection(
            config_df.index
        )
    )

    assert len(common_seeds) == 50


    for metric in [
        "average_reward",
        "DP_gap",
        "EO_gap",
    ]:

        base_values = (
            baseline.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )

        config_values = (
            config_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )


        # Positive = better
        if metric == "average_reward":

            delta = (
                config_values
                - base_values
            )

        else:

            delta = (
                base_values
                - config_values
            )


        n = len(delta)

        mean_delta = (
            delta.mean()
        )

        sd_delta = (
            delta.std(
                ddof=1
            )
        )

        ci95 = (
            t.ppf(
                0.975,
                df=n - 1,
            )
            * sd_delta
            / np.sqrt(n)
        )


        delta_rows.append(
            {
                "configuration":
                    configuration,

                "metric":
                    metric,

                "mean_delta":
                    mean_delta,

                "ci95":
                    ci95,
            }
        )


adult_ablation_delta_df = (
    pd.DataFrame(
        delta_rows
    )
)


# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 6),
    sharey=True,
)


PANEL_INFO = [

    (
        "average_reward",
        "A. Average reward",
        "Δ reward vs no intervention",
    ),

    (
        "DP_gap",
        "B. Demographic parity",
        "Reduction in DP gap vs no intervention",
    ),

    (
        "EO_gap",
        "C. Equalized odds",
        "Reduction in EO gap vs no intervention",
    ),
]


y_positions = np.arange(
    len(CONFIG_ORDER)
)


for ax, (
    metric,
    title,
    xlabel,
) in zip(
    axes,
    PANEL_INFO,
):

    panel = (
        adult_ablation_delta_df[
            adult_ablation_delta_df[
                "metric"
            ]
            == metric
        ]
        .set_index(
            "configuration"
        )
        .loc[
            CONFIG_ORDER
        ]
    )


    ax.errorbar(
        panel[
            "mean_delta"
        ],
        y_positions,

        xerr=panel[
            "ci95"
        ],

        fmt="o",

        capsize=3,
    )


    ax.axvline(
        0,
        linewidth=1,
    )


    ax.set_title(
        title
    )

    ax.set_xlabel(
        xlabel
    )

    ax.grid(
        True,
        alpha=0.20,
    )


axes[0].set_yticks(
    y_positions
)

axes[0].set_yticklabels(
    CONFIG_ORDER
)

axes[0].invert_yaxis()


fig.suptitle(
    "Adult Income: ablation of preprocessing, "
    "in-processing and post-processing",
    fontsize=14,
)


fig.text(
    0.5,
    0.01,
    (
        "Differences are paired by seed relative to the no-intervention "
        "configuration. Positive values indicate improvement. "
        "Points show mean differences across 50 seeds and error bars "
        "represent 95% confidence intervals."
    ),
    ha="center",
    fontsize=9,
)


fig.tight_layout(
    rect=[
        0,
        0.05,
        1,
        0.95,
    ]
)


ABLATION_FIG_DIR = (
    FIG_DIR
    / "ablation"
)

ABLATION_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


ABLATION_PNG_PATH = (
    ABLATION_FIG_DIR
    / "adult_linucb_full_ablation_3panel.png"
)

ABLATION_SVG_PATH = (
    ABLATION_FIG_DIR
    / "adult_linucb_full_ablation_3panel.svg"
)


fig.savefig(
    ABLATION_PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    ABLATION_SVG_PATH,
    bbox_inches="tight",
)


plt.show()


print(
    "Saved PNG:",
    ABLATION_PNG_PATH,
)

print(
    "Saved SVG:",
    ABLATION_SVG_PATH,
)

In [ ]:
# ============================================================
# Adult — 3-panel factorial ablation figure
#
# A. Change in average reward vs no intervention
# B. Reduction in demographic parity gap vs no intervention
# C. Reduction in equalized odds gap vs no intervention
#
# IMPORTANT:
# Positive values ALWAYS indicate improvement.
#
# 95% CI are calculated on PAIRED seed-level differences.
# ============================================================



# ------------------------------------------------------------
# 1. Reload seed-level ablation data if necessary
# ------------------------------------------------------------

ABLATION_SEEDLEVEL_PATH = (
    TABLE_DIR
    / "adult_linucb_full_ablation_seedlevel.csv"
)

if "adult_ablation_df" not in globals():

    adult_ablation_df = pd.read_csv(
        ABLATION_SEEDLEVEL_PATH
    )

print(
    "Ablation seed-level data:",
    adult_ablation_df.shape
)


# ------------------------------------------------------------
# 2. Configuration order
# ------------------------------------------------------------

CONFIG_ORDER = [
    "Pre only",
    "In only",
    "Post only",
    "Pre + In",
    "Pre + Post",
    "In + Post",
    "Pre + In + Post",
]


# ------------------------------------------------------------
# 3. Baseline = no fairness intervention
# ------------------------------------------------------------

baseline = (
    adult_ablation_df[
        adult_ablation_df[
            "configuration"
        ].astype(str)
        == "None"
    ]
    .set_index("seed")
    .sort_index()
)

assert len(baseline) == 50


# ------------------------------------------------------------
# 4. Calculate paired differences
#
# Reward:
#     configuration - baseline
#
# DP / EO:
#     baseline - configuration
#
# Therefore:
#     positive = improvement
#     negative = deterioration
# ------------------------------------------------------------

delta_rows = []


for configuration in CONFIG_ORDER:

    config_df = (
        adult_ablation_df[
            adult_ablation_df[
                "configuration"
            ].astype(str)
            == configuration
        ]
        .set_index("seed")
        .sort_index()
    )

    common_seeds = (
        baseline.index
        .intersection(
            config_df.index
        )
    )

    assert len(common_seeds) == 50


    for metric in [
        "average_reward",
        "DP_gap",
        "EO_gap",
    ]:

        baseline_values = (
            baseline.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )

        config_values = (
            config_df.loc[
                common_seeds,
                metric,
            ]
            .to_numpy(
                dtype=float
            )
        )


        # --------------------------------------------
        # Positive values always mean improvement
        # --------------------------------------------

        if metric == "average_reward":

            paired_difference = (
                config_values
                - baseline_values
            )

        else:

            paired_difference = (
                baseline_values
                - config_values
            )


        n = len(
            paired_difference
        )

        mean_difference = (
            paired_difference.mean()
        )

        sd_difference = (
            paired_difference.std(
                ddof=1
            )
        )

        t_crit = t.ppf(
            0.975,
            df=n - 1,
        )

        ci95 = (
            t_crit
            * sd_difference
            / np.sqrt(n)
        )


        delta_rows.append(
            {
                "configuration":
                    configuration,

                "metric":
                    metric,

                "mean_difference":
                    mean_difference,

                "ci95":
                    ci95,

                "ci_low":
                    mean_difference
                    - ci95,

                "ci_high":
                    mean_difference
                    + ci95,

                "n":
                    n,
            }
        )


adult_ablation_delta_df = pd.DataFrame(
    delta_rows
)


# ------------------------------------------------------------
# 5. Display numerical values used in the figure
# ------------------------------------------------------------

display(
    adult_ablation_delta_df
    .pivot(
        index="configuration",
        columns="metric",
        values="mean_difference",
    )
    .loc[
        CONFIG_ORDER
    ]
    .round(4)
)


# ------------------------------------------------------------
# 6. Create 3-panel figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 6.2),
    sharey=True,
)


PANEL_INFO = [

    (
        "average_reward",
        "A. Predictive utility",
        "Change in average reward",
    ),

    (
        "DP_gap",
        "B. Demographic parity",
        "Reduction in DP gap",
    ),

    (
        "EO_gap",
        "C. Equalized odds",
        "Reduction in EO gap",
    ),
]


y_positions = np.arange(
    len(CONFIG_ORDER)
)


for ax, (
    metric,
    title,
    xlabel,
) in zip(
    axes,
    PANEL_INFO,
):

    panel_df = (
        adult_ablation_delta_df[
            adult_ablation_delta_df[
                "metric"
            ]
            == metric
        ]
        .set_index(
            "configuration"
        )
        .loc[
            CONFIG_ORDER
        ]
    )


    ax.errorbar(
        panel_df[
            "mean_difference"
        ].to_numpy(),

        y_positions,

        xerr=panel_df[
            "ci95"
        ].to_numpy(),

        fmt="o",

        markersize=6,

        capsize=3,

        linewidth=1.2,
    )


    # Reference = no difference from baseline
    ax.axvline(
        0,
        linewidth=1,
        linestyle="--",
    )


    ax.set_title(
        title,
        fontsize=11,
    )

    ax.set_xlabel(
        xlabel,
        fontsize=10,
    )

    ax.grid(
        True,
        axis="x",
        alpha=0.20,
    )


# ------------------------------------------------------------
# 7. Configuration labels
# ------------------------------------------------------------

axes[0].set_yticks(
    y_positions
)

axes[0].set_yticklabels(
    CONFIG_ORDER,
    fontsize=9,
)

axes[0].invert_yaxis()


# ------------------------------------------------------------
# 8. Global title and explanatory note
# ------------------------------------------------------------

fig.suptitle(
    (
        "Adult Income — factorial ablation of preprocessing, "
        "in-processing and post-processing"
    ),
    fontsize=14,
)


fig.text(
    0.5,
    0.01,
    (
        "Effects are paired differences relative to the no-intervention "
        "configuration across 50 seeds. "
        "Positive values indicate improvement; negative values indicate deterioration. "
        "Error bars represent 95% confidence intervals of the paired differences."
    ),
    ha="center",
    fontsize=9,
)


fig.tight_layout(
    rect=[
        0,
        0.06,
        1,
        0.94,
    ]
)


# ------------------------------------------------------------
# 9. Save figure
# ------------------------------------------------------------

ABLATION_FIG_DIR = (
    FIG_DIR
    / "ablation"
)

ABLATION_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


ABLATION_PNG_PATH = (
    ABLATION_FIG_DIR
    / "adult_linucb_full_ablation_3panel.png"
)

ABLATION_SVG_PATH = (
    ABLATION_FIG_DIR
    / "adult_linucb_full_ablation_3panel.svg"
)


fig.savefig(
    ABLATION_PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    ABLATION_SVG_PATH,
    bbox_inches="tight",
)


plt.show()


print(
    "Saved PNG:",
    ABLATION_PNG_PATH,
)

print(
    "Saved SVG:",
    ABLATION_SVG_PATH,
)

## Longitudinal post-processing benchmark

This optional section evaluates held-out post-processing at several training horizons.

It is used only to produce the post-processing-over-horizon figure. The main endpoint summaries and significance tests are still based on the final horizon.

In [ ]:
POSTPROC_HORIZONS = [
    500,
    1000,
    2000,
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
]

POSTPROC_HORIZON_DIR = RUN_DIR / "postprocessing_horizons"
POSTPROC_HORIZON_PATH = (POSTPROC_HORIZON_DIR / "adult_postprocessing_over_horizon.csv")

RUN_POSTPROC_HORIZON_BENCHMARK = False

POSTPROC_HORIZON_FAMILIES = ["linucb", "linear_ts", "exp4",]

if RUN_POSTPROC_HORIZON_BENCHMARK:
    horizon_rows = []

    for horizon in POSTPROC_HORIZONS:
        print("\nPost-processing horizon:", horizon)

        horizon_params = replace(params, t_max=int(horizon),)

        for family in POSTPROC_HORIZON_FAMILIES:
            family_config = POLICY_FAMILIES[family]
            family_horizon_dir = (POSTPROC_HORIZON_DIR / family / f"horizon_{horizon}")

            _, _, postproc_horizon_df, _ = run_adult_family_benchmark(
                family=family,
                run_dir=family_horizon_dir,
                X_train=adult_data.X_train,
                y_train=adult_data.y_train,
                g_train=adult_data.g_train,
                X_cal=adult_data.X_cal,
                y_cal=adult_data.y_cal,
                g_cal=adult_data.g_cal,
                X_test=adult_data.X_test,
                y_test=adult_data.y_test,
                g_test=adult_data.g_test,
                advice_train=ADVICE_TRAIN if family == "exp4" else None,
                advice_cal=ADVICE_CAL if family == "exp4" else None,
                advice_test=ADVICE_TEST if family == "exp4" else None,
                preprocessings=PREPROCESSINGS,
                policies=family_config["policies"],
                seeds=SEEDS,
                params=horizon_params,
                force_rerun=FORCE_RERUN,
            )

            postproc_horizon_df = postproc_horizon_df.copy()
            postproc_horizon_df["horizon"] = int(horizon)
            postproc_horizon_df["t"] = int(horizon)

            horizon_rows.append(postproc_horizon_df)

    adult_postproc_horizon_df = normalize_metric_columns(
        pd.concat(
            horizon_rows,
            ignore_index=True,
        )
    )

    POSTPROC_HORIZON_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    adult_postproc_horizon_df.to_csv(
        POSTPROC_HORIZON_PATH,
        index=False,
    )

    print("Saved:", POSTPROC_HORIZON_PATH)
    print("adult_postproc_horizon_df:", adult_postproc_horizon_df.shape)

else:
    if POSTPROC_HORIZON_PATH.exists():
        adult_postproc_horizon_df = normalize_metric_columns(
            pd.read_csv(POSTPROC_HORIZON_PATH)
        )

        print("Loaded:", POSTPROC_HORIZON_PATH)
        print("adult_postproc_horizon_df:", adult_postproc_horizon_df.shape)

    else:
        adult_postproc_horizon_df = pd.DataFrame()

        print(
            "No longitudinal post-processing file found. "
            "The post-processing-over-horizon figure will be skipped."
        )

## Figures

This section generates the standard Adult-sex figure set for each policy family.

For each family, the notebook produces:

- a preprocessing comparison for Demographic Parity Gap and Equalized Odds Gap;
- an in-processing comparison under uniform preprocessing;
- temporal utility figures for average reward, cumulative prediction error, and UtilityGap;
- a two-panel post-processing comparison over the training horizon;
- final predictive and fairness dot plots;
- the post-processing figure combines the online baseline reference with held-out fairness-aware and post-processed evaluations when longitudinal data are available.

The visual convention is kept consistent across families: baseline policies are blue, fairness-aware policies are red, post-processed policies are green, uniform preprocessing is shown with solid lines, and reweighting with dotted lines. Temporal bands indicate pointwise 95% confidence intervals for the mean trajectory across seeds.

In [ ]:
figure_paths = []

for family, family_config in POLICY_FAMILIES.items():
    figure_paths.extend(
        plot_adult_family_figure_set(
            temporal_df=adult_temporal_df,
            family=family,
            family_label=family_config["label"],
            policies=family_config["policies"],
            preprocessings=PREPROCESSINGS,
            fig_dir=FIG_DIR,
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            show=True,
        )
    )

    figure_paths.extend(
        plot_adult_final_dot_plots(
            endpoint_df=adult_endpoint_df,
            postproc_df=adult_postproc_df,
            family=family,
            family_label=family_config["label"],
            policies=family_config["policies"],
            preprocessings=PREPROCESSINGS,
            fig_dir=FIG_DIR,
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            show=True,
        )
    )

    if not adult_postproc_horizon_df.empty:
        baseline_policy, fair_policy = family_config["policies"]

        family_postproc_horizon_df = adult_postproc_horizon_df[
            adult_postproc_horizon_df["family"] == family
        ].copy()

        if not family_postproc_horizon_df.empty:
            figure_paths.append(
                plot_adult_postprocessing_over_horizon(
                    postproc_horizon_df=family_postproc_horizon_df,
                    temporal_df=adult_temporal_df,
                    family=family,
                    family_label=family_config["label"],
                    baseline_policy=baseline_policy,
                    fair_policy=fair_policy,
                    horizons=POSTPROC_HORIZONS,
                    preprocessings=PREPROCESSINGS,
                    fig_dir=FIG_DIR,
                    policy_labels=POLICY_LABELS,
                    preprocessing_labels=PREPROCESSING_LABELS,
                    show=True,
                )
            )
        else:
            print(
                "Skipping post-processing-over-horizon figure for",
                family,
                "because no longitudinal post-processing data are available.",
            )

# Focused LinUCB in-processing average-reward figure
figure_paths.append(
    plot_adult_linucb_inprocessing_average_reward(
        temporal_df=adult_temporal_df,
        fig_dir=FIG_DIR,
        policy_labels=POLICY_LABELS,
        show=True,
    )
)

# Focused LinUCB post-processing average-reward figure
if not adult_postproc_horizon_df.empty:
    figure_paths.append(
        plot_adult_linucb_postprocessing_average_reward(
            postproc_horizon_df=adult_postproc_horizon_df,
            fig_dir=FIG_DIR,
            policy_labels=POLICY_LABELS,
            show=True,
        )
    )
else:
    print(
        "Skipping focused Adult LinUCB post-processing average-reward figure "
        "because adult_postproc_horizon_df is empty."
    )

print("Generated figures:", len(figure_paths))

for path in figure_paths:
    print(path)

## Trade-off plots

In [ ]:
ADULT_TRADEOFF_DIR = FIG_DIR / "tradeoff"

adult_tradeoff_paths = plot_real_dataset_tradeoff_set(
    dataset_label="ADULT",
    sensitive_label="sex",
    endpoint_df=adult_endpoint_df,
    postproc_df=adult_postproc_df,
    policy_families=POLICY_FAMILIES,
    preprocessings=PREPROCESSINGS,
    fig_dir=ADULT_TRADEOFF_DIR,
    file_prefix="adult_sex",
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    confidence=0.95,
    show=True,
)

print("Adult trade-off figures:", len(adult_tradeoff_paths))
for path in adult_tradeoff_paths:
    print(path)

## Final summary tables

This section exports compact final summary tables for the Adult-sex experiments.

The main utility table follows the thesis convention: average reward, cumulative prediction error, and UtilityGap are reported as the primary compact metrics. Fairness-specific metrics are exported separately when needed.

In [ ]:
summary_tables = export_adult_final_summary_tables(
    endpoint_df=adult_endpoint_df,
    postproc_df=adult_postproc_df,
    table_dir=TABLE_DIR,
    families=POLICY_FAMILIES,
    preprocessings=PREPROCESSINGS,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
)

## Paired significance tests

This section performs paired statistical comparisons across seeds.

The comparisons are defined separately for each policy family and cover the three intervention levels:

- preprocessing: baseline policy under uniform weighting versus reweighting;
- in-processing: baseline policy versus fairness-aware policy;
- post-processing: fairness-aware policy before versus after group-specific threshold calibration.

The tests are paired by seed and corrected for multiple comparisons using Holm correction.

In [ ]:
SIGNIFICANCE_COMPARISONS_BY_FAMILY = {
    "linucb": [
        {
            "Comparison": "Preprocessing: LinUCB uniform vs reweighting",
            "baseline_policy": "LinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "LinUCB",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: LinUCB vs FairLinUCB",
            "baseline_policy": "LinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinUCB",
            "intervention_preprocessing": "uniform",
        },
        {
            "Comparison": "Post-processing: FairLinUCB vs FairLinUCB+PP",
            "baseline_policy": "FairLinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinUCB+PP",
            "intervention_preprocessing": "uniform",
        },
    ],
    "linear_ts": [
        {
            "Comparison": "Preprocessing: LinTS uniform vs reweighting",
            "baseline_policy": "LinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "LinTS",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: LinTS vs FairLinTS",
            "baseline_policy": "LinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinTS",
            "intervention_preprocessing": "uniform",
        },
        {
            "Comparison": "Post-processing: FairLinTS vs FairLinTS+PP",
            "baseline_policy": "FairLinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinTS+PP",
            "intervention_preprocessing": "uniform",
        },
    ],
    "exp4": [
        {
            "Comparison": "Preprocessing: EXP4 uniform vs reweighting",
            "baseline_policy": "EXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "EXP4",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: EXP4 vs FairEXP4",
            "baseline_policy": "EXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairEXP4",
            "intervention_preprocessing": "uniform",
        },
        {
            "Comparison": "Post-processing: FairEXP4 vs FairEXP4+PP",
            "baseline_policy": "FairEXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairEXP4+PP",
            "intervention_preprocessing": "uniform",
        },
    ],
}

significance_tables = export_adult_significance_tables(
    endpoint_df=adult_endpoint_df,
    postproc_df=adult_postproc_df,
    table_dir=TABLE_DIR,
    families=POLICY_FAMILIES,
    comparisons_by_family=SIGNIFICANCE_COMPARISONS_BY_FAMILY,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
)

## Generated artifacts

This final section lists the figures, LaTeX/CSV tables, and raw cached outputs generated by the notebook.

In [ ]:
print("FIGURES")
for path in sorted(FIG_DIR.glob("adult_sex_*.png")):
    print(path.name)

print("\nTABLES")
for path in sorted(TABLE_DIR.glob("adult_sex_*")):
    print(path.name)

print("\nRAW RUN OUTPUTS")
for family in POLICY_FAMILIES:
    family_run_dir = RUN_DIR / family
    print("\nFamily:", family)
    for name in ["temporal.csv", "endpoint.csv", "postprocessing.csv", "postprocessing_parameters.csv"]:
        path = family_run_dir / name
        print(name, "exists =", path.exists(), "path =", path)

In [ ]:
from pathlib import Path

RUN_DIR = Path("results/full/adult_sex_cmab/full")
FIG_DIR = Path("results/full/adult_sex_cmab/final_figures")
TABLE_DIR = Path("results/full/adult_sex_cmab/overleaf_tables")

print("RUN_DIR:", RUN_DIR)
print("RF exists:", (RUN_DIR / "random_forest_endpoint.csv").exists())

In [ ]:
import pandas as pd

# Recharge directement les résultats déjà calculés
adult_postproc_df = pd.concat(
    [
        pd.read_csv(RUN_DIR / "linucb" / "postprocessing.csv"),
        pd.read_csv(RUN_DIR / "linear_ts" / "postprocessing.csv"),
        pd.read_csv(RUN_DIR / "exp4" / "postprocessing.csv"),
    ],
    ignore_index=True,
)

adult_rf_df = pd.read_csv(
    RUN_DIR / "random_forest_endpoint.csv"
)

print("Bandit post-processing:", adult_postproc_df.shape)
print("Random Forest:", adult_rf_df.shape)

print("\nPolicies:")
print(adult_postproc_df["policy"].unique())

In [ ]:
# ============================================================
# Adult: supervised Random Forest baseline vs fairness-aware
# contextual bandits on the SAME held-out test set
# ============================================================


# 1) Load Random Forest results
RF_PATH = RUN_DIR / "random_forest_endpoint.csv"
adult_rf_df = pd.read_csv(RF_PATH)
# 2) Keep only the fairness-aware bandit policies BEFORE post-processing.
# These rows in adult_postproc_df are evaluated on the held-out test set,
# therefore they are directly comparable with the Random Forest.
FAIR_TEST_POLICIES = ["FairLinUCB", "FairLinTS", "FairEXP4"]

adult_supervised_comparison_df = pd.concat(
    [
        adult_rf_df[adult_rf_df["policy"] == "RandomForest"].copy(),
        adult_postproc_df[
            adult_postproc_df["policy"].isin(FAIR_TEST_POLICIES)
        ].copy(),
    ],
    ignore_index=True,
)

# Safety checks
assert adult_supervised_comparison_df["seed"].nunique() == 50
assert set(adult_supervised_comparison_df["preprocessing"]) == {
    "uniform",
    "reweigh_group_label",
}

# 3) Mean and 95% Student-t CI across seeds
METRICS = ["average_reward", "DP_gap", "EO_gap"]

summary_rows = []

for (policy, preprocessing), group in adult_supervised_comparison_df.groupby(
    ["policy", "preprocessing"]
):
    n = len(group)
    t_crit = t.ppf(0.975, df=n - 1)

    row = {
        "policy": policy,
        "preprocessing": preprocessing,
        "n": n,
    }

    for metric in METRICS:
        mean = group[metric].mean()
        sd = group[metric].std(ddof=1)
        ci95 = t_crit * sd / np.sqrt(n)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci95"] = ci95
        row[f"{metric}_low"] = mean - ci95
        row[f"{metric}_high"] = mean + ci95

    summary_rows.append(row)

adult_supervised_summary_df = pd.DataFrame(summary_rows)

POLICY_ORDER = ["RandomForest", "FairLinUCB", "FairLinTS", "FairEXP4"]
PREP_ORDER = ["uniform", "reweigh_group_label"]

adult_supervised_summary_df["policy"] = pd.Categorical(
    adult_supervised_summary_df["policy"],
    categories=POLICY_ORDER,
    ordered=True,
)
adult_supervised_summary_df["preprocessing"] = pd.Categorical(
    adult_supervised_summary_df["preprocessing"],
    categories=PREP_ORDER,
    ordered=True,
)

adult_supervised_summary_df = adult_supervised_summary_df.sort_values(
    ["policy", "preprocessing"]
).reset_index(drop=True)

display(
    adult_supervised_summary_df[
        [
            "policy",
            "preprocessing",
            "average_reward_mean",
            "average_reward_ci95",
            "DP_gap_mean",
            "DP_gap_ci95",
            "EO_gap_mean",
            "EO_gap_ci95",
        ]
    ].round(4)
)

# Save the numerical table
TABLE_DIR.mkdir(parents=True, exist_ok=True)
SUPERVISED_TABLE_PATH = TABLE_DIR / "adult_rf_vs_fair_bandits_summary.csv"
adult_supervised_summary_df.to_csv(SUPERVISED_TABLE_PATH, index=False)
print("Saved:", SUPERVISED_TABLE_PATH)


# 4) Trade-off figures
COMPARISON_FIG_DIR = FIG_DIR / "supervised_baseline"
COMPARISON_FIG_DIR.mkdir(parents=True, exist_ok=True)

SHORT_LABELS = {
    ("RandomForest", "uniform"): "RF-U",
    ("RandomForest", "reweigh_group_label"): "RF-RW",
    ("FairLinUCB", "uniform"): "FairLinUCB-U",
    ("FairLinUCB", "reweigh_group_label"): "FairLinUCB-RW",
    ("FairLinTS", "uniform"): "FairLinTS-U",
    ("FairLinTS", "reweigh_group_label"): "FairLinTS-RW",
    ("FairEXP4", "uniform"): "FairEXP4-U",
    ("FairEXP4", "reweigh_group_label"): "FairEXP4-RW",
}

OFFSETS_DP = {
    "RF-U": (6, 6),
    "RF-RW": (6, 6),
    "FairLinUCB-U": (-55, -14),
    "FairLinUCB-RW": (6, 6),
    "FairLinTS-U": (6, 7),
    "FairLinTS-RW": (-8, -15),
    "FairEXP4-U": (-72, 7),
    "FairEXP4-RW": (6, 7),
}

OFFSETS_EO = {
    "RF-U": (6, 6),
    "RF-RW": (6, 6),
    "FairLinUCB-U": (6, 7),
    "FairLinUCB-RW": (6, 7),
    "FairLinTS-U": (6, 7),
    "FairLinTS-RW": (6, 7),
    "FairEXP4-U": (-58, 7),
    "FairEXP4-RW": (6, 7),
}


def plot_supervised_tradeoff(
    summary_df,
    fairness_metric,
    fairness_label,
    filename,
    offsets,
):
    fig, ax = plt.subplots(figsize=(8.2, 5.8))

    for _, row in summary_df.iterrows():
        policy = str(row["policy"])
        preprocessing = str(row["preprocessing"])
        label = SHORT_LABELS[(policy, preprocessing)]

        marker = "o" if preprocessing == "uniform" else "s"

        ax.errorbar(
            row[f"{fairness_metric}_mean"],
            row["average_reward_mean"],
            xerr=row[f"{fairness_metric}_ci95"],
            yerr=row["average_reward_ci95"],
            fmt=marker,
            markersize=6,
            capsize=3,
        )

        dx, dy = offsets[label]
        ax.annotate(
            label,
            (
                row[f"{fairness_metric}_mean"],
                row["average_reward_mean"],
            ),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=8.5,
        )

    ax.set_xlabel(f"{fairness_label} (lower is better)")
    ax.set_ylabel("Average reward / accuracy (higher is better)")
    ax.set_title(
        "Adult (sex): supervised baseline vs fairness-aware contextual bandits"
    )
    ax.grid(True, alpha=0.22)

    fig.tight_layout()

    path = COMPARISON_FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", path)
    return path


adult_rf_dp_path = plot_supervised_tradeoff(
    adult_supervised_summary_df,
    fairness_metric="DP_gap",
    fairness_label="Demographic parity gap",
    filename="adult_rf_vs_fair_bandits_dp.png",
    offsets=OFFSETS_DP,
)

adult_rf_eo_path = plot_supervised_tradeoff(
    adult_supervised_summary_df,
    fairness_metric="EO_gap",
    fairness_label="Equalized odds gap",
    filename="adult_rf_vs_fair_bandits_eo.png",
    offsets=OFFSETS_EO,
)


# ============================================================
# Optional: paired Wilcoxon tests, Random Forest vs each
# fairness-aware bandit, with Holm correction
# ============================================================

from scipy.stats import wilcoxon

comparisons = []

for preprocessing in PREP_ORDER:
    rf_seed = (
        adult_supervised_comparison_df[
            (adult_supervised_comparison_df["policy"] == "RandomForest")
            & (adult_supervised_comparison_df["preprocessing"] == preprocessing)
        ]
        .set_index("seed")
        .sort_index()
    )

    for policy in FAIR_TEST_POLICIES:
        bandit_seed = (
            adult_supervised_comparison_df[
                (adult_supervised_comparison_df["policy"] == policy)
                & (adult_supervised_comparison_df["preprocessing"] == preprocessing)
            ]
            .set_index("seed")
            .sort_index()
        )

        common_seeds = rf_seed.index.intersection(bandit_seed.index)

        for metric in ["average_reward", "DP_gap", "EO_gap"]:
            statistic, p_value = wilcoxon(
                rf_seed.loc[common_seeds, metric],
                bandit_seed.loc[common_seeds, metric],
                alternative="two-sided",
            )

            comparisons.append(
                {
                    "preprocessing": preprocessing,
                    "comparison": f"RandomForest vs {policy}",
                    "metric": metric,
                    "mean_difference_RF_minus_bandit": (
                        rf_seed.loc[common_seeds, metric]
                        - bandit_seed.loc[common_seeds, metric]
                    ).mean(),
                    "p_raw": p_value,
                }
            )

adult_rf_significance_df = pd.DataFrame(comparisons)

# Holm correction without statsmodels
p_values = adult_rf_significance_df["p_raw"].to_numpy()
m = len(p_values)

order = np.argsort(p_values)
sorted_p = p_values[order]

adjusted_sorted = np.empty(m)

for i, p in enumerate(sorted_p):
    adjusted_sorted[i] = min((m - i) * p, 1.0)

# Holm adjusted p-values must be monotonically non-decreasing
adjusted_sorted = np.maximum.accumulate(adjusted_sorted)

p_holm = np.empty(m)
p_holm[order] = adjusted_sorted

adult_rf_significance_df["p_holm"] = p_holm
adult_rf_significance_df["significant_0.05"] = (
    adult_rf_significance_df["p_holm"] < 0.05
)


display(adult_rf_significance_df)

RF_SIGNIFICANCE_PATH = (
    TABLE_DIR / "adult_rf_vs_fair_bandits_wilcoxon_holm.csv"
)
adult_rf_significance_df.to_csv(RF_SIGNIFICANCE_PATH, index=False)
print("Saved:", RF_SIGNIFICANCE_PATH)


In [ ]:
# ============================================================
# ADULT — sensitivity analysis for lambda_DP
#
# FairLinUCB + uniform preprocessing
# 10 lambda values × 50 seeds
#
# ONLINE endpoint metrics at T = 30,000
# Existing Adult benchmark is NOT modified.
# Results are cached after every run.
# ============================================================



from fair_bandits.experiments.adult_runner import (
    run_adult_family_trajectory,
)


# ------------------------------------------------------------
# 1. Same lambda grid as COMPAS
# ------------------------------------------------------------

LAMBDA_GRID = [
    0.0,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
    8.0,
    16.0,
    32.0,
    64.0,
]

SENSITIVITY_SEEDS = list(
    range(50)
)


# ------------------------------------------------------------
# 2. Separate cache directory
# ------------------------------------------------------------

LAMBDA_SENSITIVITY_DIR = (
    RUN_DIR
    / "lambda_dp_sensitivity"
)

LAMBDA_SENSITIVITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LAMBDA_RESULTS_PATH = (
    LAMBDA_SENSITIVITY_DIR
    / "adult_fairlinucb_lambda_dp_50seeds.csv"
)


# ------------------------------------------------------------
# 3. Reload previous partial results
# ------------------------------------------------------------

if LAMBDA_RESULTS_PATH.exists():

    lambda_adult_results_df = pd.read_csv(
        LAMBDA_RESULTS_PATH
    )

else:

    lambda_adult_results_df = pd.DataFrame()


# ------------------------------------------------------------
# 4. Detect already completed runs
# ------------------------------------------------------------

done = set()


if not lambda_adult_results_df.empty:

    done = set(
        zip(
            lambda_adult_results_df[
                "lambda_dp"
            ].astype(float),

            lambda_adult_results_df[
                "seed"
            ].astype(int),
        )
    )


total = (
    len(LAMBDA_GRID)
    * len(SENSITIVITY_SEEDS)
)


print(
    f"Already complete: "
    f"{len(done)}/{total}"
)


# ------------------------------------------------------------
# 5. Run Adult FairLinUCB sensitivity
# ------------------------------------------------------------

for lambda_dp in LAMBDA_GRID:

    sensitivity_params = replace(
        params,
        dp_lambda=float(
            lambda_dp
        ),
    )


    for seed in SENSITIVITY_SEEDS:

        key = (
            float(lambda_dp),
            int(seed),
        )


        if key in done:

            print(
                "Cached:",
                key,
            )

            continue


        print(
            "Running Adult:",
            "lambda =",
            lambda_dp,
            "| seed =",
            seed,
        )


        trajectory_df, _ = (
            run_adult_family_trajectory(

                family="linucb",

                X_train=
                    adult_data.X_train,

                y_train=
                    adult_data.y_train,

                g_train=
                    adult_data.g_train,

                advice_train=None,

                seed=int(seed),

                policy_name=
                    "FairLinUCB",

                preprocessing=
                    "uniform",

                params=
                    sensitivity_params,
            )
        )


        # --------------------------------------------
        # Final ONLINE checkpoint
        # --------------------------------------------

        final_row = (
            trajectory_df
            .sort_values("t")
            .iloc[-1]
        )


        new_row = {

            "dataset":
                "Adult",

            "lambda_dp":
                float(lambda_dp),

            "seed":
                int(seed),

            "t":
                int(
                    final_row["t"]
                ),

            "average_reward":
                float(
                    final_row[
                        "average_reward"
                    ]
                ),

            "DP_gap":
                float(
                    final_row[
                        "DP_gap"
                    ]
                ),

            "EO_gap":
                float(
                    final_row[
                        "EO_gap"
                    ]
                ),
        }


        lambda_adult_results_df = pd.concat(
            [
                lambda_adult_results_df,

                pd.DataFrame(
                    [new_row]
                ),
            ],
            ignore_index=True,
        )


        # --------------------------------------------
        # Save after EACH run
        # --------------------------------------------

        lambda_adult_results_df.to_csv(
            LAMBDA_RESULTS_PATH,
            index=False,
        )


        done.add(
            key
        )


        print(
            f"Completed: "
            f"{len(done)}/{total}"
        )


print(
    "\nSensitivity analysis complete:",
    len(done),
    "/",
    total,
)

print(
    "Saved:",
    LAMBDA_RESULTS_PATH,
)

In [ ]:
# ============================================================
# ADULT — lambda_DP sensitivity summary
# Mean + 95% Student-t CI across 50 seeds
# ============================================================



# Reload if needed
if "lambda_adult_results_df" not in globals():

    lambda_adult_results_df = pd.read_csv(
        LAMBDA_RESULTS_PATH
    )


summary_rows = []


for lambda_dp, group in (
    lambda_adult_results_df
    .groupby(
        "lambda_dp"
    )
):

    n = len(group)

    row = {

        "lambda_dp":
            float(lambda_dp),

        "n":
            n,
    }


    for metric in [
        "average_reward",
        "DP_gap",
        "EO_gap",
    ]:

        mean = (
            group[
                metric
            ].mean()
        )

        sd = (
            group[
                metric
            ].std(
                ddof=1
            )
        )

        ci95 = (
            t.ppf(
                0.975,
                df=n - 1,
            )
            * sd
            / np.sqrt(n)
        )


        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95


    summary_rows.append(
        row
    )


adult_lambda_summary_df = (
    pd.DataFrame(
        summary_rows
    )

    .sort_values(
        "lambda_dp"
    )

    .reset_index(
        drop=True
    )
)


display(
    adult_lambda_summary_df
    .round(4)
)


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

assert (
    adult_lambda_summary_df[
        "n"
    ]
    == 50
).all()


# ------------------------------------------------------------
# Save definitive summary
# ------------------------------------------------------------

ADULT_LAMBDA_SUMMARY_PATH = (
    TABLE_DIR
    / "adult_fairlinucb_lambda_dp_sensitivity_summary.csv"
)


adult_lambda_summary_df.to_csv(
    ADULT_LAMBDA_SUMMARY_PATH,
    index=False,
)


print(
    "Saved:",
    ADULT_LAMBDA_SUMMARY_PATH,
)